In [ ]:
# @title 1.1 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev libtbb-dev libtbbmalloc2

# libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
!apt-get install -y -qq libcgal-dev libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.2 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml shapely mip POT


In [ ]:
%%bash
# @title 1.3 Installation de HGP-clusterer
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi



In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# @title 1.4 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

print("Configuration CGAL...")

# -- FIX: Add CGAL path to environment for setup_cgal.py --
cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
# ---------------------------------------------------------

# On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
# car le setup.py principal pourrait réussir autrement.
try:
    subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
except subprocess.CalledProcessError:
    print("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
# Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
# Note: On le passe aussi explicitement ici pour être sûr
install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")


# Maya benchmark

Clustering with HGP-Clusterer on the Maya dataset.

In [ ]:
# @title 2.1 Load, Normalize and Cluster
normalization = "L2" # @param ["L2", "L1", "Mahalanobis", "None"]
n_components = 4 # @param {type:"slider", min:1, max:4, step:1}

import numpy as np
import pandas as pd
from hgp_clusterer import HGPClusterer
from sklearn.decomposition import PCA

# Load data from local parquet
df = pd.read_parquet('tests/Maya/simu_MB_10Myr.parquet')
features = ['eccentricity', 'inclination', 'absolute_magnitude', 'semi_major_axis']
X = df[features].values.astype(float)

# Apply chosen normalization / transformation
if normalization == "L2":
    X = X / np.linalg.norm(X, axis=0, keepdims=True)
elif normalization == "L1":
    X = X / np.linalg.norm(X, ord=1, axis=0, keepdims=True)
elif normalization == "Mahalanobis":
    # Whitening via PCA equals transforming variables by the inverse square root of the covariance matrix.
    # This means the Euclidean distance in the new space corresponds exactly to the Mahalanobis distance.
    n_comp = min(n_components, X.shape[1])
    pca = PCA(n_components=n_comp, whiten=True)
    X = pca.fit_transform(X)
    features = [f"PC{i+1}" for i in range(n_comp)]
elif normalization == "None":
    pass # No normalization

print(f"Data shape: {X.shape}")

import sys
import time
print('------------------------------------------------------')
print(' [VERBOSE] Initialisation de HGPClusterer...')
print(f'   -> Backend sélectionné : CGAL')
print(f'   -> K = 1')
print(f'   -> Dimension des données = {X.shape[1]}')
print('------------------------------------------------------')
sys.stdout.flush()

clusterer = HGPClusterer(backend='cgal', K=1, verbose=True, min_cluster_size=15, expZ=4)
print("Running fit_predict...")
print('\n[VERBOSE] Lancement de fit_predict()...')
sys.stdout.flush()

start_time = time.time()
labels = clusterer.fit_predict(X)
print(f'\n[VERBOSE] fit_predict() terminé en {time.time() - start_time:.2f} secondes !')
sys.stdout.flush()

print(f"Found clusters: {np.unique(labels)}")
counts = np.bincount(labels[labels >= 0])
if len(counts) > 0:
    print(f"Cluster sizes: {counts}")
print(f"Noise points: {np.sum(labels == -1)}")


In [ ]:
# @title 2.2 Plot Results (Semi-Major Axis vs Eccentricity)
import plotly.express as px
import pandas as pd

# We use the original dataframe for unnormalized plotting
plot_df = df.copy()
plot_df['Cluster'] = labels
plot_df['Cluster_Label'] = plot_df['Cluster'].astype(str)
plot_df.loc[plot_df['Cluster'] == -1, 'Cluster_Label'] = 'Noise'

fig = px.scatter(
    plot_df,
    x='semi_major_axis',
    y='eccentricity',
    color='Cluster_Label',
    hover_data={'Cluster': True, 'Cluster_Label': False},
    color_discrete_map={'Noise': 'lightgrey'},
    title='HGP-Clusterer Results on Maya Dataset',
    labels={'semi_major_axis': 'Semi-Major Axis (AU)', 'eccentricity': 'Eccentricity'}
)
fig.update_traces(marker=dict(size=3, opacity=0.8))
fig.show()


In [ ]:
# @title 2.3 Plot All Combinations (Normalized Coordinates)
import plotly.express as px
import pandas as pd
from itertools import combinations

# Les variables 'features', 'X', et 'labels' proviennent des cellules précédentes
df_norm = pd.DataFrame(X, columns=features)
df_norm['Cluster'] = labels
df_norm['Cluster_Label'] = df_norm['Cluster'].astype(str)
df_norm.loc[df_norm['Cluster'] == -1, 'Cluster_Label'] = 'Noise'

# Génère automatiquement les combinaisons pour n'importe quel nombre de features
for f1, f2 in combinations(features, 2):
    fig = px.scatter(
        df_norm,
        x=f1,
        y=f2,
        color='Cluster_Label',
        hover_data={'Cluster': True, 'Cluster_Label': False},
        color_discrete_map={'Noise': 'lightgrey'},
        title=f'Normalized Coordinates: {f2} vs {f1}'
    )
    fig.update_traces(marker=dict(size=3, opacity=0.8))
    fig.show()
